In [1]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.tools import Tool
import requests

In [2]:
load_dotenv()

True

In [3]:
llm  = ChatGroq(
    model = 'llama-3.3-70b-versatile'
)

In [24]:
from langchain_core.tools import tool
from ddgs import DDGS  # Changed from duckduckgo_search

@tool
def search_tool(query: str) -> str:
    """Search the web using DuckDuckGo for live, up-to-date information."""
    # Initialize using the new ddgs class structure
    with DDGS() as scraper:
        results = [r for r in scraper.text(query, max_results=2)]
        if not results:
            return "No results found."
        
        # Format the output into a clean string for your agent
        return "\n\n".join([
            f"Title: {r.get('title')}\nURL: {r.get('href')}\nSnippet: {r.get('body')}" 
            for r in results
        ])


In [39]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=e906d973f72dfcf408038982f664e39d&query={city}'

  response = requests.get(url)

  return response.json()

In [40]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt="You are a helpful assistant. Use tools when needed."
)

response = agent.invoke({"messages": [{"role": "user", "content": "Find the capital of Punjab(pakistan), then find its current weather condition"}]})

In [53]:
print(response['messages'][-1].content)

The capital of Punjab, Pakistan is Lahore. The current weather condition in Lahore is sunny with a temperature of 38 degrees Celsius and a humidity of 39%. The wind speed is 11 km/h, and the wind direction is ESE. The atmospheric pressure is 996 mbar, and the precipitation is 0 mm. The feels-like temperature is 41 degrees Celsius, and the UV index is 0. The visibility is 10 km, and it is daytime.
